# Revela — Notebook 02: ISIC 2019 Metadata Inspection

**Issue:** B1.3 — Inspect metadata fields  
**Dataset:** ISIC 2019 Training Dataset  
**Files:**
- `ISIC_2019_Training_GroundTruth.csv`
- `ISIC_2019_Training_Metadata.csv`

**Covers:**
- Available columns
- Missing values summary
- Raw class distribution
- Skin-tone metadata availability

## Block 1 — Load Files

In [ ]:
import pandas as pd

gt = pd.read_csv('../../revela/ISIC_2019_Training_GroundTruth.csv')
meta = pd.read_csv('../../revela/ISIC_2019_Training_Metadata.csv')

print(f"Ground truth rows: {len(gt)}")
print(f"Metadata rows:     {len(meta)}")
print(f"\nGround truth columns: {gt.columns.tolist()}")
print(f"Metadata columns:     {meta.columns.tolist()}")

## Block 2 — Column Types and Missing Values

In [ ]:
print("=== GROUND TRUTH DTYPES ===")
print(gt.dtypes)

print("\n=== GROUND TRUTH MISSING VALUES ===")
print(gt.isnull().sum())

print("\n=== METADATA DTYPES ===")
print(meta.dtypes)

print("\n=== METADATA MISSING VALUES ===")
print(meta.isnull().sum())

## Block 3 — Raw Class Distribution

In [ ]:
classes = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

print("=== CLASS DISTRIBUTION (ISIC 2019) ===")
total = len(gt)
for c in classes:
    count = int((gt[c] == 1.0).sum())
    pct = (count / total) * 100
    bar = '█' * int(pct / 2)
    print(f"{c:5s}: {count:5d} ({pct:.1f}%) {bar}")

## Block 4 — Skin-Tone Metadata Availability

In [ ]:
print("=== SKIN-TONE METADATA CHECK ===")
skin_tone_cols = [c for c in meta.columns if any(k in c.lower() for k in ['skin', 'fitzpatrick', 'tone', 'fst'])]

if skin_tone_cols:
    print(f"Skin-tone columns found: {skin_tone_cols}")
else:
    print("❌ No skin-tone metadata available in ISIC 2019.")
    print("   Fields available:", meta.columns.tolist())

print("\n=== AVAILABLE PATIENT METADATA ===")
print(meta[['age_approx', 'anatom_site_general', 'sex']].describe(include='all'))

## Block 5 — Body Location Distribution

In [ ]:
print("=== ANATOMICAL SITE DISTRIBUTION ===")
print(meta['anatom_site_general'].value_counts(dropna=False))

print("\n=== SEX DISTRIBUTION ===")
print(meta['sex'].value_counts(dropna=False))

print("\n=== AGE DISTRIBUTION ===")
print(meta['age_approx'].describe())

## Block 6 — Merge and Cross-Check

In [ ]:
merged = gt.merge(meta, on='image', how='inner')

print(f"Ground truth rows:  {len(gt)}")
print(f"Metadata rows:      {len(meta)}")
print(f"Successfully merged: {len(merged)}")
print(f"Unmatched rows:     {len(gt) - len(merged)}")

print("\nMerged sample:")
print(merged.head(3).to_string())

## Block 9 — Summary

| Criteria | ISIC 2019 | FitzPatrick17k |
|----------|-----------|----------------|
| Total images | 25,331 | 16,577 |
| Columns listed | ✅ | ✅ |
| Missing values summarized | ✅ | ✅ |
| Raw class distribution | ✅ | ✅ |
| MVP 3-class mapping | ✅ | ✅ |
| Skin-tone metadata | ❌ Not available | ✅ fitzpatrick_scale |
| Body location | ✅ (anatom_site_general) | ❌ Not available |

### ISIC 2019 → MVP Mapping
| Raw Class | MVP Class | Count |
|-----------|-----------|-------|
| MEL | Melanoma | 4,522 |
| NV | Benign nevus | 12,875 |
| BCC, AK, BKL, DF, VASC, SCC, UNK | Other lesion | 7,934 |

### FitzPatrick17k → MVP Mapping
| Raw Label Pattern | MVP Class |
|-------------------|-----------|
| melanoma, superficial spreading melanoma, malignant melanoma | Melanoma (~490) |
| nevus, nevi variants | Benign nevus (~485) |
| All other conditions | Other lesion (~15,602) |

**Key finding:** ISIC 2019 has no skin-tone metadata — use FitzPatrick17k for skin-tone slice evaluation. FitzPatrick17k has severe class imbalance for the MVP taxonomy (Other lesion dominates at ~94%). Neither dataset alone is sufficient — the model (D1.2) was trained on BCN20000.

In [ ]:
# MVP 3-class taxonomy (per D1.2 scope update)
# Melanoma | Benign nevus | Other lesion

mvp_mapping = {
    'MEL':  'Melanoma',
    'NV':   'Benign nevus',
    'BCC':  'Other lesion',
    'AK':   'Other lesion',
    'BKL':  'Other lesion',
    'DF':   'Other lesion',
    'VASC': 'Other lesion',
    'SCC':  'Other lesion',
    'UNK':  'Other lesion',
}

raw_classes = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
gt['raw_class'] = gt[raw_classes].idxmax(axis=1)
gt['mvp_class'] = gt['raw_class'].map(mvp_mapping)

print("=== RAW → MVP CLASS MAPPING ===")
for raw, mvp in mvp_mapping.items():
    count = int((gt['raw_class'] == raw).sum())
    print(f"  {raw:5s} → {mvp:15s}  ({count:,} images)")

print("\n=== MVP CLASS DISTRIBUTION (ISIC 2019) ===")
total = len(gt)
for cls, count in gt['mvp_class'].value_counts().items():
    pct = (count / total) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cls:15s}: {count:6,} ({pct:.1f}%) {bar}")

## Block 8 — 3-Class MVP Taxonomy Mapping (FitzPatrick17k)

In [ ]:
fp = pd.read_csv('../data/fitzpatrick17k.csv')

def map_fp_to_mvp(label):
    label = str(label).lower()
    if 'melanoma' in label:
        return 'Melanoma'
    if any(k in label for k in ['nevus', 'nevi']):
        return 'Benign nevus'
    return 'Other lesion'

fp['mvp_class'] = fp['label'].apply(map_fp_to_mvp)

print("=== MVP CLASS DISTRIBUTION (FitzPatrick17k) ===")
total = len(fp)
for cls, count in fp['mvp_class'].value_counts().items():
    pct = (count / total) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cls:15s}: {count:6,} ({pct:.1f}%) {bar}")

print("\n=== MELANOMA SUB-LABELS ===")
print(fp[fp['mvp_class'] == 'Melanoma']['label'].value_counts().to_string())

print("\n=== BENIGN NEVUS SUB-LABELS ===")
print(fp[fp['mvp_class'] == 'Benign nevus']['label'].value_counts().to_string())

## Block 7 — Summary

| Criteria | ISIC 2019 |
|----------|-----------|
| Total images | 25,331 |
| Columns listed | ✅ |
| Missing values summarized | ✅ |
| Raw class distribution | ✅ |
| Skin-tone metadata | ❌ Not available |
| Body location | ✅ (anatom_site_general, 2,631 missing) |
| Diagnosis source | ✅ (one-hot labels in GroundTruth CSV) |

**Key finding:** ISIC 2019 has no skin-tone metadata. Fairness evaluation by skin tone is not possible with this dataset alone — use FitzPatrick17k for skin-tone slice evaluation (see `01_inspect_FP.ipynb`).